In [0]:
!pip install lightgbm

In [0]:
!pip install shap

In [0]:
import os
import re
import gc
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds

import lightgbm as lgb
import matplotlib.pyplot as plt
import psutil

In [0]:
# ============================================================
# Paths
# ============================================================

DATA_PATH = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "feature_selection_mdb_25k_final_parquet"
)

OUTPUT_DIR = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/v2/"
    "lgbm_shap_dependency_outputs"
)

# Directly loading model/mapping from current notebook directory
MODEL_PATH = "lightgbm_nbrx_model_wo_segment_new_V8.txt"

FEATURE_MAPPING_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/feature_alias_mapping_shap_V8.csv"

LOCAL_OUTPUT_DIR = "/tmp/lgbm_pdp_outputs"
Path(LOCAL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ============================================================
# PDP feature
# Give exact feature name here
# ============================================================

PDP_FEATURE_NAME = "UNMET_NEED_COUNT_P3"

# ============================================================
# PDP settings
# ============================================================

# Use all rows
USE_ALL_ROWS = True

# Number of PDP grid points
PDP_GRID_SIZE = 25

# Use quantiles to avoid extreme outlier values
LOWER_QUANTILE = 0.01
UPPER_QUANTILE = 0.99

# Read batch size.
# Since you are using large memory compute, 4096 is reasonable.
# If memory issue occurs, reduce to 2048.
ARROW_BATCH_SIZE = 4096

# LightGBM CPU threads
NUM_THREADS = psutil.cpu_count(logical=False)

if NUM_THREADS is None:
    NUM_THREADS = 64

NUM_THREADS = min(NUM_THREADS, 64)

# Target name only used for plot labels
TARGET_NAME_FOR_LABEL = "TOTAL_NBRX_C4"

print("DATA_PATH:", DATA_PATH)
print("MODEL_PATH:", MODEL_PATH)
print("FEATURE_MAPPING_PATH:", FEATURE_MAPPING_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PDP_FEATURE_NAME:", PDP_FEATURE_NAME)
print("NUM_THREADS:", NUM_THREADS)

In [0]:
# ============================================================
# Helper functions
# ============================================================

def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))[:180]


def copy_local_to_output(local_path, output_filename=None):
    """
    Copy local file from /tmp to Databricks Volume output folder.
    """
    local_path = str(local_path)

    if output_filename is None:
        output_filename = os.path.basename(local_path)

    final_output_path = f"{OUTPUT_DIR.rstrip('/')}/{output_filename}"

    try:
        dbutils.fs.cp(
            f"file:{local_path}",
            f"dbfs:{final_output_path}",
            True
        )
    except Exception:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local_path, final_output_path)

    print("Saved:")
    print(final_output_path)

    return final_output_path


def get_cast_schema_if_needed(dataset, columns):
    """
    Returns a float32 cast schema only when needed.
    If parquet is already float32, returns None to avoid extra casting.
    """
    schema = dataset.schema
    needs_cast = False

    for col in columns:
        dtype = schema.field(col).type

        if not pa.types.is_float32(dtype):
            needs_cast = True
            break

    if needs_cast:
        return pa.schema([(col, pa.float32()) for col in columns])

    return None


def table_to_float32_numpy(table, cast_schema=None):
    """
    Convert Arrow table to float32 NumPy array.
    """
    if cast_schema is not None:
        table = table.cast(cast_schema, safe=False)

    pdf_batch = table.to_pandas(
        split_blocks=True,
        self_destruct=True
    )

    arr = pdf_batch.to_numpy(
        dtype=np.float32,
        copy=True
    )

    del pdf_batch

    # LightGBM handles NaN as missing.
    # Replace inf with NaN.
    bad_mask = np.isinf(arr)

    if bad_mask.any():
        arr[bad_mask] = np.nan

    return arr


def build_pdp_grid(values, grid_size=25, lower_q=0.01, upper_q=0.99, include_zero=True):
    """
    Build PDP grid using quantiles of actual feature values.
    """
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError("No finite values found for PDP feature.")

    unique_values = np.unique(values)

    if len(unique_values) <= grid_size:
        grid = unique_values
    else:
        q = np.linspace(lower_q, upper_q, grid_size)
        grid = np.nanquantile(values, q)
        grid = np.unique(grid.astype(np.float32))

    if include_zero and np.nanmin(values) <= 0 <= np.nanmax(values):
        grid = np.unique(
            np.concatenate([
                grid,
                np.array([0.0], dtype=np.float32)
            ])
        )

    grid = np.sort(grid.astype(np.float32))

    return grid

In [0]:
# ============================================================
# Load saved LightGBM model
# ============================================================

booster = lgb.Booster(model_file=MODEL_PATH)

print("Model loaded successfully.")
print("Number of model features:", booster.num_feature())
print("Number of trees:", booster.num_trees())
print("Current iteration:", booster.current_iteration())

# ============================================================
# Load feature mapping
# ============================================================

feature_mapping_df = pd.read_csv(FEATURE_MAPPING_PATH)

required_cols = {"feature_index", "feature_alias", "feature"}

missing_cols = required_cols - set(feature_mapping_df.columns)

if len(missing_cols) > 0:
    raise ValueError(f"Feature mapping file missing columns: {missing_cols}")

feature_mapping_df = (
    feature_mapping_df
    .sort_values("feature_index")
    .reset_index(drop=True)
)

feature_cols = feature_mapping_df["feature"].tolist()
feature_aliases = feature_mapping_df["feature_alias"].tolist()

n_features = len(feature_cols)

print("Number of features in mapping:", n_features)

if booster.num_feature() != n_features:
    raise ValueError(
        f"Model expects {booster.num_feature()} features, "
        f"but feature mapping has {n_features} features."
    )

if PDP_FEATURE_NAME not in feature_cols:
    possible_matches = [
        c for c in feature_cols
        if PDP_FEATURE_NAME.lower() in c.lower()
    ]

    print("PDP feature not found exactly.")
    print("Possible matches:")
    for m in possible_matches[:50]:
        print(m)

    raise ValueError(f"PDP feature not found in model features: {PDP_FEATURE_NAME}")

PDP_FEATURE_INDEX = feature_cols.index(PDP_FEATURE_NAME)

print("PDP feature found:")
print(PDP_FEATURE_NAME)

print("PDP feature index:")
print(PDP_FEATURE_INDEX)

In [0]:
# ============================================================
# Read PyArrow dataset
# ============================================================

dataset = ds.dataset(DATA_PATH, format="parquet")

all_cols = dataset.schema.names

missing_features = [
    c for c in feature_cols
    if c not in all_cols
]

if len(missing_features) > 0:
    print("Missing feature count:", len(missing_features))
    print("Sample missing features:")
    print(missing_features[:20])
    raise ValueError("Some model features are missing from the parquet dataset.")

n_rows_total = dataset.count_rows()

print("Total rows in parquet:", n_rows_total)
print("Using all rows for PDP:", USE_ALL_ROWS)
print("Total model features:", n_features)

In [0]:
# ============================================================
# Build PDP grid from all rows
# Reads only PDP feature column
# ============================================================

feature_cast_schema = get_cast_schema_if_needed(
    dataset=dataset,
    columns=[PDP_FEATURE_NAME]
)

scanner_feature = dataset.scanner(
    columns=[PDP_FEATURE_NAME],
    batch_size=ARROW_BATCH_SIZE,
    use_threads=True
)

feature_value_blocks = []

rows_seen = 0
start_time = time.time()

for batch_id, record_batch in enumerate(scanner_feature.to_batches(), start=1):
    table = pa.Table.from_batches([record_batch])

    values_arr = table_to_float32_numpy(
        table=table,
        cast_schema=feature_cast_schema
    )

    values = values_arr[:, 0]

    feature_value_blocks.append(values)

    rows_seen += record_batch.num_rows

    if batch_id % 20 == 0 or rows_seen == n_rows_total:
        elapsed = time.time() - start_time
        print(
            f"Grid pass rows read: {rows_seen:,} / {n_rows_total:,} "
            f"| elapsed sec: {elapsed:,.1f}"
        )

    del table, values_arr, values
    gc.collect()

feature_values_for_grid = np.concatenate(feature_value_blocks)

pdp_grid = build_pdp_grid(
    values=feature_values_for_grid,
    grid_size=PDP_GRID_SIZE,
    lower_q=LOWER_QUANTILE,
    upper_q=UPPER_QUANTILE,
    include_zero=True
)

print("PDP grid values:")
print(pdp_grid)

print("Number of grid points:", len(pdp_grid))

In [0]:
# ============================================================
# Compute PDP using all rows
# ============================================================

scan_cols = feature_cols

full_cast_schema = get_cast_schema_if_needed(
    dataset=dataset,
    columns=scan_cols
)

scanner_full = dataset.scanner(
    columns=scan_cols,
    batch_size=ARROW_BATCH_SIZE,
    use_threads=True
)

pdp_pred_sum = np.zeros(len(pdp_grid), dtype=np.float64)
pdp_pred_count = np.zeros(len(pdp_grid), dtype=np.int64)

rows_processed = 0
start_time = time.time()

for batch_id, record_batch in enumerate(scanner_full.to_batches(), start=1):
    table = pa.Table.from_batches([record_batch])

    X_batch = table_to_float32_numpy(
        table=table,
        cast_schema=full_cast_schema
    )

    batch_rows = X_batch.shape[0]

    # For each PDP grid value, replace only the PDP feature column.
    # Other features remain exactly as observed.
    for grid_idx, grid_value in enumerate(pdp_grid):
        X_batch[:, PDP_FEATURE_INDEX] = grid_value

        preds = booster.predict(
            X_batch,
            num_threads=NUM_THREADS
        )

        finite_mask = np.isfinite(preds)

        pdp_pred_sum[grid_idx] += float(np.sum(preds[finite_mask]))
        pdp_pred_count[grid_idx] += int(np.sum(finite_mask))

        del preds

    rows_processed += batch_rows

    if batch_id % 10 == 0 or rows_processed == n_rows_total:
        elapsed = time.time() - start_time
        print(
            f"PDP rows processed: {rows_processed:,} / {n_rows_total:,} "
            f"| elapsed sec: {elapsed:,.1f}"
        )

    del table, X_batch
    gc.collect()

avg_prediction = pdp_pred_sum / pdp_pred_count

pdp_df = pd.DataFrame({
    "feature": PDP_FEATURE_NAME,
    "feature_value": pdp_grid,
    "avg_prediction": avg_prediction,
    "rows_used": n_rows_total,
    "grid_points": len(pdp_grid)
})

pdp_df["centered_avg_prediction"] = (
    pdp_df["avg_prediction"] - pdp_df["avg_prediction"].mean()
)

print("PDP calculation complete.")
pdp_df

In [0]:
# ============================================================
# Save PDP data
# ============================================================

safe_feature = safe_filename(PDP_FEATURE_NAME)

local_pdp_csv_path = os.path.join(
    LOCAL_OUTPUT_DIR,
    f"pdp_data_{safe_feature}_all_rows.csv"
)

pdp_df.to_csv(local_pdp_csv_path, index=False)

pdp_csv_output_path = copy_local_to_output(
    local_pdp_csv_path,
    f"pdp_data_{safe_feature}_all_rows.csv"
)

print("PDP data saved to:")
print(pdp_csv_output_path)

In [0]:
# ============================================================
# PDP plot
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    pdp_df["feature_value"],
    pdp_df["avg_prediction"],
    marker="o",
    linewidth=2
)

plt.xlabel(PDP_FEATURE_NAME)
plt.ylabel(f"Average predicted {TARGET_NAME_FOR_LABEL}")
plt.title(f"PDP Plot: {PDP_FEATURE_NAME}")
plt.grid(True, alpha=0.3)
plt.tight_layout()

local_pdp_plot_path = os.path.join(
    LOCAL_OUTPUT_DIR,
    f"pdp_plot_{safe_feature}_all_rows.png"
)

plt.savefig(
    local_pdp_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

pdp_plot_output_path = copy_local_to_output(
    local_pdp_plot_path,
    f"pdp_plot_{safe_feature}_all_rows.png"
)

print("PDP plot saved to:")
print(pdp_plot_output_path)